# Distribution of profiles in ATLAS database

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
species = "Escherichia_coli" # write the species name with a "_" instead of a space, e.g., "Escherichia_coli"

df = pd.read_csv(f'/Users/jimenamartinreina/Documents/PabloCatalan/DATABASES/VivliDataChallenge/Vivli_{species}.csv')

# read Claude database to repeat figures
#df = pd.read_csv(f'/Users/jimenamartinreina/Documents/PabloCatalan/DATABASES/VivliDataChallenge/claudedb.csv')

## Rank-frequency graph to show heavy tailed distribution in ATLAS database

**Count and frequency**

List of antibiotics:

Fluoroquinolones: ciprofloxacin, levofloxacin

Carbapenems: meropenem, imipenem

Cephalosporins: ceftazidime, cefepime, ceftaroline

Penicillins + BLI combos: ampicillin, ampicillin-sulbactam, amoxicillin-clavulanate, piperacillin-tazobactam

Monobactam: aztreonam

Aminoglycosides: amikacin, gentamicin

Folate: trimethoprim-sulfa

In [ ]:
# resistance_columns = [col for col in df.columns if col.endswith('_I') and col != 'Colistin_I']

# only 15 antibiotics (because of small ratio R/S)

resistance_columns = ["Ciprofloxacin_I", "Levofloxacin_I", "Meropenem_I", "Imipenem_I", "Ceftazidime_I",
                      "Cefepime_I", "Ceftaroline_I", "Ampicillin_I", "Ampicillin sulbactam_I", 
                      "Amoxycillin clavulanate_I", "Piperacillin tazobactam_I", "Aztreonam_I", "Amikacin_I", 
                      "Gentamicin_I", "Trimethoprim sulfa_I"]

# We do not include Colistin, as it is always Resistant, it does not add any value
df_resistance = df[resistance_columns]
df_resistance_profiles = df_resistance.drop_duplicates()

# count how many times each row from df_resistance_profiles appears in df_resistance
profile_counts = df_resistance.groupby(df_resistance_profiles.columns.tolist()).size().reset_index(name='count')

# get relative frequency for each row
profile_counts['frequency'] = profile_counts['count'] / profile_counts['count'].sum()

**Label**

In [ ]:
eucast_df = pd.read_excel('/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/data/Antimicrobial_abbreviations_v7_20220120.xlsx', sheet_name='LIST')
eucast_dict = (eucast_df[['Agents', 'Abbreviation']]
               .dropna(subset=['Agents', 'Abbreviation'])
               .assign(Agents=lambda x: x['Agents'].str.strip().str.upper())
               .set_index('Agents')['Abbreviation']
               .to_dict())

# Aliases: some names differ between ATLAS and EUCAST
aliases = {
    'AMOXYCILLIN CLAVULANATE':    'AMOXICILLIN-CLAVULANIC ACID',
    'AMPICILLIN SULBACTAM':       'AMPICILLIN-SULBACTAM',
    'CEFOPERAZONE SULBACTAM':     'CEFOPERAZONE-SULBACTAM',
    'CEFTAZIDIME AVIBACTAM':      'CEFTAZIDIME/AVIBACTAM',
    'PIPERACILLIN TAZOBACTAM':    'PIPERACILLIN/TAZOBACTAM',
    'TRIMETHOPRIM SULFA':         'TRIMETHOPRIM-SULFAMETHOXAZOLE',
}

eucast_dict['AZTREONAM AVIBACTAM'] = 'AZT-AVI'  # does not appear in EUCAST, found in literature

# Build acronym list from antibiotic column names
antibiotics = [col for col in resistance_columns if col.replace('_I', '').strip().upper() in eucast_dict or col.replace('_I', '').strip().upper() in aliases]

def get_eucast_acronym(col_name):
    clean = col_name.replace('_I', '').strip().upper()
    # Apply alias if needed
    lookup = aliases.get(clean, clean)
    if lookup in eucast_dict:
        return eucast_dict[lookup]
    print(f"WARNING: '{clean}' not found in EUCAST, using generated acronym")
    parts = clean.split()
    if len(parts) > 1:
        return parts[0][:4] + "-" + parts[1][:2]
    return parts[0][:4]
acronyms = [get_eucast_acronym(a) for a in antibiotics]

# # Verify — print the mapping
# for ab, acr in zip(antibiotics, acronyms):
#     print(f"  {ab.replace('_I',''):40} -> {acr}")

def create_label(profile, acronyms):
    resis = [acronyms[i] for i, value in enumerate(profile) if value == "Resistant"]
    return "+".join(resis) if resis else "Susceptible"

In [ ]:
for profile in profile_counts[resistance_columns].values:
    label = create_label(profile, acronyms)
    profile_counts.loc[(profile_counts[resistance_columns] == profile).all(axis=1), 'label'] = label

**num_resistant**

In [ ]:
# count number of resistances ( == Resistant) in each profile
profile_counts['num_resistant'] = (profile_counts[resistance_columns] == "Resistant").sum(axis=1)

**Rank-frequency plot**

In [ ]:
profile_totals = profile_counts.groupby('label')['count'].sum().sort_values(ascending=False).reset_index()
profile_totals['rank'] = range(1, len(profile_totals) + 1)
profile_totals['rel_freq'] = profile_totals['count'] / profile_totals['count'].sum() * 100
num_res = profile_counts.drop_duplicates('label').set_index('label')['num_resistant']
profile_totals['num_resistant'] = profile_totals['label'].map(num_res)

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.cm.RdYlGn_r
norm = plt.Normalize(vmin=0, vmax=profile_totals['num_resistant'].max())
sc = ax.scatter(profile_totals['rank'], profile_totals['rel_freq'],
                c=profile_totals['num_resistant'], cmap=cmap, norm=norm,
                s=8, alpha=0.7, linewidths=0)

# Anotating labels
for _, row in profile_totals.head(5).iterrows():
    short_label = row['label'] if len(row['label']) < 50 else row['label'][:27] + '…' # truncate very long labels
    ax.annotate(short_label, xy=(row['rank'], row['rel_freq']),
                xytext=(row['rank'] + 15, row['rel_freq'] * 0.85),
                fontsize=8, color='#333333',
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.6))

cbar = fig.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Number of resistances', fontsize=9)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Rank (log scale)', fontsize=11)
ax.set_ylabel('Relative frequency (%, log scale)', fontsize=11)
# ax.set_title(f'Rank–frequency distribution of resistance profiles in {species.replace("_", " ")}\n(ATLAS 2018–2024, all isolates combined)',
#              fontsize=15, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

total = len(profile_totals)
rare = (profile_totals['count'] <= 3).sum()
ax.text(0.55, 0.85, f"{rare} profiles ({rare/total*100:.0f}%) appear\n≤3 times across all years",
        transform=ax.transAxes, fontsize=13, color='gray',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='lightgray', alpha=0.9))

plt.tight_layout()
# Save the figure because the quality is much better
plt.savefig(f'/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/outputs/rank_frequency/rank_frequency_{species}.png', dpi=300, bbox_inches='tight')
plt.savefig(f'/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/outputs/figures/rank_frequency_{species}.png', dpi=600, bbox_inches='tight')
plt.show()

## Number of profiles for each layer (0 resistances, 1 resistance, etc.)

In [ ]:
import math
import matplotlib.pyplot as plt

# 1. Configuración de datos
n = 19
x = list(range(n + 1))
y_absolute = [math.comb(n, k) for k in x]
total_profiles = sum(y_absolute)

# Convertir a porcentajes
y_percentage = [(val / total_profiles) * 100 for val in y_absolute]

# 2. Configuración de la figura (Alta resolución para papers)
fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

# 3. Diseño del gráfico de barras
bars = ax.bar(x, y_percentage, color='#2C5282', edgecolor='#1A365D', alpha=0.9, width=0.75, zorder=3)

# 4. Etiquetas y textos en inglés (Porcentaje)
ax.set_xlabel('Number of Resistant Antibiotics (k)', fontsize=12, fontweight='bold', labelpad=12)
ax.set_ylabel('Percentage of Theoretical Profiles (%)', fontsize=12, fontweight='bold', labelpad=12)

# Configuración de los ejes
ax.set_xticks(x)
ax.set_xticklabels(x, fontsize=10)

# Formatear el eje Y con el símbolo % y un decimal
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda val, loc: f"{val:.1f}%"))
ax.tick_params(axis='y', labelsize=10)

# 5. Estética profesional (Estilo Clean/Minimalista)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CBD5E0')
ax.spines['bottom'].set_color('#CBD5E0')

ax.grid(axis='y', linestyle='--', alpha=0.5, color='#E2E8F0', zorder=0)
ax.set_axisbelow(True)

# 6. Cuadro de texto interno
info_text = f"Total Antibiotics (n) = {n}\nTotal Combinatorial Space = {total_profiles:,}"
ax.text(0.05, 0.92, info_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontweight='semibold', color='#2D3748',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#F7FAFC', edgecolor='#E2E8F0', alpha=0.9))

# 7. Ajuste y guardado
plt.tight_layout()
#plt.savefig('theoretical_profiles_percentage.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Asignamos el resultado de tu groupby
counts = profile_counts.groupby('num_resistant').size()

x = counts.index
y_absolute = counts.values
total_observed = counts.sum()

# Convertir tus datos a porcentajes
y_percentage = (y_absolute / total_observed) * 100

# 2. Configuración de la figura
fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

# 3. Diseño del gráfico de barras
bars = ax.bar(x, y_percentage, color='#2C5282', edgecolor='#1A365D', alpha=0.9, width=0.75, zorder=3)

# 4. Etiquetas en inglés adaptadas a porcentaje
ax.set_xlabel('Number of Resistant Antibiotics (k)', fontsize=12, fontweight='bold', labelpad=12)
ax.set_ylabel('Percentage of Observed Profiles (%)', fontsize=12, fontweight='bold', labelpad=12)

# Configuración del eje X
ax.set_xticks(x)
ax.set_xticklabels(x, fontsize=10)

# Formatear el eje Y con el símbolo % y un decimal
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda val, loc: f"{val:.1f}%"))
ax.tick_params(axis='y', labelsize=10)

# 5. Estética profesional
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CBD5E0')
ax.spines['bottom'].set_color('#CBD5E0')

ax.grid(axis='y', linestyle='--', alpha=0.5, color='#E2E8F0', zorder=0)
ax.set_axisbelow(True)

# 6. Cuadro de texto interno con los metadatos de tu muestra
info_text = f"Total Antibiotics (n) = {len(x)-1}\nTotal Observed Profiles = {total_observed:,}"
ax.text(0.05, 0.92, info_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontweight='semibold', color='#2D3748',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#F7FAFC', edgecolor='#E2E8F0', alpha=0.9))

# 7. Ajuste y guardado
plt.tight_layout()
#plt.savefig('observed_profiles_percentage.png', dpi=300, bbox_inches='tight')
plt.show()

## Null model

In [ ]:
profile_counts

In [ ]:
# group profile_touns by "num_resistant" and then sum the "count" column for each group
layer_counts = profile_counts.groupby("num_resistant")["count"].sum()

In [ ]:
layer_counts

In [ ]:
"""
Modelo nulo de ocupacion para el espacio de perfiles de resistencia
=====================================================================

Pregunta que responde:
Para cada capa k (aislados resistentes a exactamente k de n antibioticos),
el numero de perfiles DISTINTOS observados en ATLAS, es menor de lo que
cabria esperar si cada aislado eligiera al azar y de forma independiente
uno de los C(n, k) perfiles posibles de esa capa?

Es el problema clasico de "bolas en cajas" (occupancy problem / coupon
collector): N bolas (aislados) lanzadas uniformemente al azar en M cajas
(perfiles posibles). Tiene solucion analitica cerrada para la media y
varianza del numero de cajas ocupadas, asi que no hace falta simular
para obtener el resultado principal -- solo para validar la aproximacion
normal en las capas con pocos aislados.

Uso:
    1. Sustituir `isolate_counts_per_k` por el numero real de AISLADOS
       (no perfiles) que caen en cada capa k
    2. Sustituir `observed_profiles_per_k` por el numero de perfiles
       DISTINTOS observados en cada capa (los datos detras de vuestra
       figura "Observed Profile Count"), ej:
           O_k_series = df.groupby("n_resistances")["profile_id"].nunique()
    3. Ejecutar el script.
"""

import numpy as np
import pandas as pd
from math import comb
from scipy import stats


# ---------------------------------------------------------------------
# 1) Modelo nulo analitico (balls-into-bins)
# ---------------------------------------------------------------------

def occupancy_moments(M, N):
    """
    Media y varianza analiticas del numero de cajas ocupadas cuando se
    lanzan N bolas de forma independiente y uniforme en M cajas.

    Usa log1p/expm1 para evitar perdida de precision quando N << M
    (que es justamente el caso mas comun aqui: pocos aislados frente a
    cientos de miles de perfiles posibles en las capas centrales).
    """
    if N == 0 or M == 0:
        return 0.0, 0.0

    if M == 1:
        # caso degenerado: solo existe un perfil posible en esta capa
        # (k=0 o k=n_antibiotics) -> con N>=1 aislados, siempre se
        # observa exactamente esa 1 caja ocupada, sin varianza.
        return 1.0, 0.0

    # log((1 - 1/M)^N) = N * log1p(-1/M)  -> mas estable que (1-1/M)**N
    log_q1 = N * np.log1p(-1.0 / M)
    p1 = np.exp(log_q1)
    one_minus_p1 = -np.expm1(log_q1)          # = 1 - (1-1/M)^N
    mean = M * one_minus_p1

    if M > 1:
        log_q2 = N * np.log1p(-2.0 / M)
        p2 = np.exp(log_q2)
    else:
        p2 = 0.0

    var = M * (M - 1) * p2 + M * p1 - (M * p1) ** 2
    var = max(var, 0.0)  # guarda frente a negativos minusculos por error de float

    return mean, var


def occupancy_zscore(observed, M, N):
    """
    Z-score y p-valor unilateral (P(X <= observado)) bajo aproximacion
    normal del modelo nulo. Un p-valor pequeno indica que se observan
    MENOS perfiles distintos de los esperados por azar -> restriccion
    (canalizacion) real del espacio de perfiles en esa capa.
    """
    mean, var = occupancy_moments(M, N)
    if var == 0:
        return np.nan, np.nan
    z = (observed - mean) / np.sqrt(var)
    p_one_sided = stats.norm.cdf(z)
    return z, p_one_sided


# ---------------------------------------------------------------------
# 2) Correccion por comparaciones multiples (Benjamini-Hochberg)
#    Necesaria porque se testan las 20 capas (k=0..19) simultaneamente.
# ---------------------------------------------------------------------

def benjamini_hochberg(p_values, alpha=0.05):
    p = np.asarray(p_values, dtype=float)
    valid = ~np.isnan(p)
    adj_p = np.full_like(p, np.nan)
    reject = np.zeros_like(p, dtype=bool)

    pv = p[valid]
    n = len(pv)
    if n > 0:
        order = np.argsort(pv)
        ranked = pv[order] * n / (np.arange(n) + 1)
        adj = np.minimum.accumulate(ranked[::-1])[::-1]
        adj = np.clip(adj, 0, 1)
        out = np.empty(n)
        out[order] = adj
        adj_p[valid] = out
        reject[valid] = out <= alpha

    return adj_p, reject


# ---------------------------------------------------------------------
# 3) Tabla comparativa completa
# ---------------------------------------------------------------------

def build_null_comparison(isolate_counts_per_k, observed_profiles_per_k, n_antibiotics=19):
    rows = []
    for k in range(n_antibiotics + 1):
        N_k = isolate_counts_per_k.get(k, 0)
        O_k = observed_profiles_per_k.get(k, 0)
        M_k = comb(n_antibiotics, k)

        mean, var = occupancy_moments(M_k, N_k)
        z, p = occupancy_zscore(O_k, M_k, N_k)

        # Aviso: si N_k >> M_k, el nulo esta practicamente saturado
        # (casi seguro que se ocupan todas las cajas) y la varianza es
        # casi cero -> el z-score pierde sentido numerico aunque el
        # p-valor (~0) siga siendo la conclusion correcta.
        nulo_saturado = (mean / M_k) > 0.999 if M_k > 0 else False

        rows.append({
            "k": k,
            "M_posibles": M_k,
            "N_aislados": N_k,
            "O_perfiles_observados": O_k,
            "E_nulo": mean,
            "SD_nulo": np.sqrt(var) if var else np.nan,
            "z_score": z,
            "p_valor_unilateral": p,
            "nulo_saturado": nulo_saturado,
        })

    df = pd.DataFrame(rows)
    adj_p, reject = benjamini_hochberg(df["p_valor_unilateral"].values, alpha=0.05)
    df["p_valor_BH"] = adj_p
    df["restriccion_significativa"] = reject
    return df


# ---------------------------------------------------------------------
# 4) Validacion Monte Carlo (opcional) -- usar solo en un par de capas
#    representativas (una de N pequeno en la cola, otra central) para
#    confirmar que la aproximacion normal es fiable.
# ---------------------------------------------------------------------

def occupancy_montecarlo(M, N, n_sims=20000, seed=0):
    if N == 0 or M == 0:
        return np.zeros(n_sims, dtype=int)
    rng = np.random.default_rng(seed)
    distinct_counts = np.empty(n_sims, dtype=int)
    for i in range(n_sims):
        draws = rng.integers(0, M, size=N)
        distinct_counts[i] = np.unique(draws).size
    return distinct_counts


# ---------------------------------------------------------------------
# Ejemplo de uso -- SUSTITUIR por vuestros datos reales
# ---------------------------------------------------------------------
if __name__ == "__main__":
    # Placeholder: sustituir por conteos reales de aislados por capa k
    # (OJO: estos numeros son un ejemplo ilustrativo, no vuestros datos
    # reales -- sustituidlos por el groupby real antes de interpretar
    # resultados)
    isolate_counts_per_k = layer_counts

    # Estos SI son los datos reales que ya teneis (bar chart "Observed
    # Profile Count" que adjuntasteis)
    observed_profiles_per_k = counts

    results = build_null_comparison(isolate_counts_per_k, observed_profiles_per_k)
    pd.set_option("display.width", 140)
    print(results.round(4).to_string(index=False))

    print("\nCapas con restriccion significativa tras correccion BH (p_BH < 0.05):")
    print(results[results["restriccion_significativa"]][
        ["k", "O_perfiles_observados", "E_nulo", "SD_nulo", "p_valor_BH"]
    ].round(3).to_string(index=False))

    # --- Validacion Monte Carlo opcional para una capa concreta ---
    k_check = 9
    M_check = comb(19, k_check)
    N_check = isolate_counts_per_k[k_check]
    mc = occupancy_montecarlo(M_check, N_check, n_sims=5000)
    mean_analitica, var_analitica = occupancy_moments(M_check, N_check)
    print(f"\nValidacion Monte Carlo para k={k_check}:")
    print(f"  Media analitica={mean_analitica:.2f}  SD analitica={np.sqrt(var_analitica):.2f}")
    print(f"  Media MC={mc.mean():.2f}  SD MC={mc.std():.2f}")

In [ ]:
"""
Figure: Observed vs. Expected Phenotypic Diversity by Resistance Layer
======================================================================

Takes the output table from `build_null_comparison()` (occupancy_null_model.py)
and plots, for each layer k, the ratio:

    O/E (%) = observed distinct profiles / expected distinct profiles
              under the uniform random assignment null model

Important note: in "saturated" layers (nulo_saturado=True, typically
very low or very high k where N_k >> M_k), E_nulo is already practically equal to
M_posibles (it is almost certain to occupy all bins by pure chance). Therefore,
the O/E ratio remains perfectly interpretable there -- what ceases to
be reliable in those layers is the z-score, not the O/E ratio -- so EVERYTHING
is plotted on the same axis, visually marking the saturated layers only
for methodological transparency.

The trivial layers k=0 and k=n_antibiotics (M=1, ratio always 100% by
definition) are excluded from the axis because they do not provide information.
"""

import pandas as pd
import plotly.graph_objects as go

def plot_occupancy_ratio_matplotlib(results_df, output_path=None):
    import matplotlib.pyplot as plt

    df = results_df.copy()
    df["OE_pct"] = 100 * df["O_perfiles_observados"] / df["E_nulo"]
    k_min, k_max = df["k"].min(), df["k"].max()
    plot_df = df[(df["k"] > k_min) & (df["k"] < k_max)].copy()

    fig, ax = plt.subplots(figsize=(9.5, 5.5))

    ax.plot(plot_df["k"], plot_df["OE_pct"], color="lightgray", lw=1.5, zorder=1)

    for saturated, color, marker, label in [
        (False, "#2E5F4E", "o", "Unsaturated null model (valid z-score)"),
        (True, "#C97B3D", "D", "Saturated null model (valid ratio; invalid z-score)"),
    ]:
        sub = plot_df[plot_df["nulo_saturado"] == saturated]
        if sub.empty:
            continue
        ax.scatter(sub["k"], sub["OE_pct"], color=color, marker=marker,
                   s=90, edgecolor="white", linewidth=0.8, zorder=2, label=label)

    ax.axhline(100, color="gray", linestyle="--", linewidth=1)
    ax.annotate("Expected under independent random assembly (100%)",
                xy=(k_min + 0.6, 101), fontsize=9, color="gray")

    for k_highlight in (1, 2):
        row = plot_df[plot_df["k"] == k_highlight]
        if row.empty:
            continue
        row = row.iloc[0]
        ax.annotate(
            f"{int(row['O_perfiles_observados'])}/{int(row['M_posibles'])} possible",
            xy=(row["k"], row["OE_pct"]),
            xytext=(row["k"] + 1.3, row["OE_pct"] + 12),
            fontsize=9,
            arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
        )

    ax.set_xlabel("Number of antibiotics to which the profile is resistant (k)")
    ax.set_ylabel("Observed / expected profiles (%)")
    ax.set_title(
        "Observed vs. Expected Phenotypic Diversity by Chance\n"
        "Observed distinct profiles as a % of what is expected under a null model\n"
        "of random assignment (ATLAS, E. coli, 2018-2024)",
        fontsize=11,
    )
    ax.set_xlim(k_min + 0.5, k_max - 0.5)
    ax.set_ylim(0, 110)
    ax.set_xticks(range(k_min + 1, k_max))
    ax.yaxis.set_major_formatter(lambda x, _: f"{int(x)}%")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
    fig.tight_layout()

    if output_path:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return fig


# ---------------------------------------------------------------------
# Example usage with your real results (the ones you already sent me)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    results = pd.DataFrame({
        "k": range(20),
        "M_posibles": [1, 19, 171, 969, 3876, 11628, 27132, 50388, 75582,
                        92378, 92378, 75582, 50388, 27132, 11628, 3876,
                        969, 171, 19, 1],
        "O_perfiles_observados": [1, 15, 42, 68, 99, 160, 176, 202, 172,
                                   163, 127, 100, 76, 51, 43, 40, 15, 8, 3, 1],
        "E_nulo": [1.0, 19.0, 171.0, 945.2044, 2060.8068, 2490.8406,
                   2257.5630, 2405.7002, 2362.3112, 2220.8801, 1848.2699,
                   1382.2143, 962.6944, 579.7276, 238.5299, 264.5724,
                   228.1426, 128.4095, 18.5191, 1.0],
        "nulo_saturado": [True, True, True, False, False, False, False,
                          False, False, False, False, False, False, False,
                          False, False, False, False, False, True],
    })

    fig = plot_occupancy_ratio_matplotlib(results, output_path="occupancy_ratio.png")
    fig.show()
